# Parent Document Retrieval in Retrieval-Augmented Generation (RAG)

When deconstructing documents for retrieval, a challenge arises:
- Smaller text chunks work better for embeddings
- But very small chunks may lose important context

**Parent Document Retrieval** solves this by:
- Storing small chunks for semantic search
- Linking each chunk to a larger parent document
- Retrieving the parent document after identifying relevant chunks


## What This Notebook Contains

This notebook demonstrates:

1. Why document chunking is required for retrieval
2. How small chunks are linked to parent documents
3. How retrieval happens using chunk embeddings
4. How the full parent document is returned for context

This implementation uses a **simple embedding simulation**
to clearly illustrate the concept.


#Imports & Setup

In [ ]:
import numpy as np
from typing import List, Dict


## Utility Functions

To keep this notebook simple and model-independent:
- A fake embedding function is used
- Cosine similarity measures semantic closeness


In [ ]:
def fake_embedding(text: str, dim: int = 64):
    """
    Generates a deterministic fake embedding for demonstration.
    """
    np.random.seed(abs(hash(text)) % (2**32))
    return np.random.rand(dim)


def cosine_similarity(v1, v2):
    return np.dot(v1, v2) / (
        np.linalg.norm(v1) * np.linalg.norm(v2)
    )


## Step 1: Parent Documents

Parent documents represent the **original full documents**
that contain complete context.


In [ ]:
parent_documents = {
    "doc_1": "Python is a high-level programming language widely used in data science, AI, and web development.",
    "doc_2": "Retrieval-Augmented Generation combines vector search with large language models to improve accuracy."
}


## Step 2: Child Chunks

Each parent document is split into **smaller chunks**.
Each chunk stores:
- Its own text
- A reference to the parent document (`parent_id`)


In [ ]:
chunks = [
    {"chunk_id": "c1", "text": "Python is a high-level programming language", "parent_id": "doc_1"},
    {"chunk_id": "c2", "text": "used in AI and data science", "parent_id": "doc_1"},
    {"chunk_id": "c3", "text": "Retrieval-Augmented Generation combines vector search", "parent_id": "doc_2"},
    {"chunk_id": "c4", "text": "with large language models", "parent_id": "doc_2"},
]


## Step 3: Embed Chunks

Embeddings are generated **only for chunks**,
not for full parent documents.


In [ ]:
for chunk in chunks:
    chunk["embedding"] = fake_embedding(chunk["text"])


## Step 4: Query-Based Retrieval

1. Convert the query into an embedding
2. Compare it with chunk embeddings
3. Select the most relevant chunk


In [ ]:
query = "What is Retrieval-Augmented Generation?"
query_embedding = fake_embedding(query)

ranked_chunks = sorted(
    chunks,
    key=lambda c: cosine_similarity(query_embedding, c["embedding"]),
    reverse=True
)

top_chunk = ranked_chunks[0]
top_chunk


## Step 5: Retrieve Parent Document

After identifying the best chunk,
its `parent_id` is used to fetch the **full document**.


In [ ]:
parent_id = top_chunk["parent_id"]
retrieved_parent_document = parent_documents[parent_id]

retrieved_parent_document


## Result & Observation

- Retrieval was performed on **small chunks**
- The final output came from the **parent document**
- This preserves full context while maintaining semantic accuracy

**Parent Document Retrieval balances:**
- Precise matching (small chunks)
- Context completeness (parent documents)
